# cop-fx-intelligence — Producto end-to-end

**Objetivo:** ver el producto completo en una sola corrida. ¿El dólar (USD/COP) **baja o no** según lo que pasó hoy?

Esto NO es el sistema de producción — es el *walking skeleton*: cada paso del pipeline ejecutado a mano para **ver el resultado** antes de montar LangGraph y la infra.

| Paso | Capa | Qué hace |
|---|---|---|
| 1 | Bronze→Silver | Noticias de hoy (CNN Colombia) |
| 2 | Gold (LLM) | Enriquecer: topic, keywords, dirección por noticia |
| 3 | EDA | Grafo relacional topics ↔ noticias |
| 4 | Bronze→Silver | Serie USD/COP, 30 días |
| 5 | — | Forecast (Prophet + ARIMA + ensemble) |
| 6 | — | Señal-noticias vs señal-serie → reconciliación |
| 7 | **Producto** | `DirectionalCall`: dirección + confianza + razonamiento |

> Cada celda tiene *fallback*: si un feed se cae o falta una API key, el notebook **igual renderiza el producto** con datos sintéticos claramente marcados. La meta es ver la forma del producto, no datos perfectos.

In [ ]:
# --- Setup: rutas e imports base -------------------------------------------
from __future__ import annotations
import sys, warnings, datetime as dt
from pathlib import Path

warnings.filterwarnings("ignore")

# Hacer importable src/ (el repo usa layout src/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np

TODAY = dt.date.today()
print(f"Repo root : {ROOT}")
print(f"src/ en path: {SRC.exists()}")
print(f"Fecha corrida: {TODAY}")

## Paso 1 — Noticias de hoy

Bajamos los titulares de CNN Español Colombia (RSS primario, HTML de respaldo). Si no hay red, usamos un set sintético para que el resto del notebook corra.

Esto es **Bronze→Silver**: dato crudo del feed, normalizado a objetos. *Sin LLM todavía.*

In [ ]:
# --- Paso 1: traer noticias ------------------------------------------------
from dataclasses import dataclass, field

@dataclass
class _FakeArticle:
    """Espejo mínimo de CNNArticle para el fallback (mismo duck-typing)."""
    title: str
    url: str
    published_at: dt.datetime
    author: str = ""
    summary: str = ""
    source: str = "sintetico"
    tags: list = field(default_factory=list)

def _synthetic_news() -> list:
    base = dt.datetime.now(dt.timezone.utc)
    raw = [
        ("BanRep mantiene la tasa de interés en su reunión de política monetaria",
         "El Banco de la República decidió mantener inalterada la tasa de intervención, "
         "citando presiones inflacionarias persistentes y la necesidad de anclar expectativas."),
        ("El precio del petróleo Brent sube ante tensiones geopolíticas",
         "Los futuros del crudo Brent avanzaron más de 2%, un viento de cola para las "
         "exportaciones colombianas y para el peso."),
        ("La Fed sugiere que mantendrá tasas altas por más tiempo",
         "Funcionarios de la Reserva Federal señalaron que el ciclo de recortes será más "
         "lento de lo esperado, fortaleciendo al dólar a nivel global."),
        ("Déficit fiscal de Colombia preocupa a los mercados",
         "El aumento del gasto público y la regla fiscal en discusión generan incertidumbre "
         "sobre la sostenibilidad de la deuda soberana."),
        ("Exportaciones colombianas crecen impulsadas por el café y el petróleo",
         "El repunte de las exportaciones mejora la balanza comercial y da soporte al peso "
         "colombiano frente al dólar."),
    ]
    return [
        _FakeArticle(title=t, url=f"https://example.com/{TODAY}/{i}",
                     published_at=base, summary=s, source="sintetico (sin red)")
        for i, (t, s) in enumerate(raw)
    ]

articles = []
source_mode = "real"
try:
    from cop_fx.data.cnn_fetcher import CNNColombiaFetcher
    articles = CNNColombiaFetcher(max_articles=15).fetch(enrich_authors=False)
    if not articles:
        raise RuntimeError("0 artículos del feed")
except Exception as exc:
    source_mode = "sintetico"
    print(f"[fallback] no se pudo usar CNNColombiaFetcher ({exc}). Uso noticias sintéticas.")
    articles = _synthetic_news()

print(f"\nModo: {source_mode}  |  {len(articles)} noticias\n")
for a in articles[:8]:
    print(f"  • {a.title[:95]}")

## Paso 2 — Enriquecimiento (Gold)

Aquí entra el primer razonamiento: cada noticia recibe **topic**, **severity** y, lo importante para nosotros, **dirección para el COP** (`bullish_cop`: ¿esta noticia tiende a *fortalecer* el peso, es decir USD/COP **baja**?).

Intentamos el `NewsAnalyzer` real (LLM **OpenAI**, `gpt-4o-mini`). Si no hay `OPENAI_API_KEY`, usamos una **heurística de keywords** determinista y gratis — la misma idea que el `_fallback_classify` del proyecto. Esto es exactamente lo que un LLM hará mejor después, pero sirve para ver el producto sin gastar un centavo.

In [ ]:
# --- Paso 2: enriquecer (topic / severity / dirección COP) -----------------
import re

TOPIC_RULES = {
    "monetary_policy": ["banrep", "banco de la república", "tasa", "fed", "reserva federal",
                        "política monetaria", "inflación", "interés"],
    "commodities":     ["petróleo", "brent", "crudo", "café", "carbón", "commodities"],
    "political_risk":  ["fiscal", "déficit", "deuda", "reforma", "gobierno", "petro", "regla fiscal"],
    "trade":           ["exportaci", "importaci", "balanza", "comercial", "aranceles"],
    "macro":           ["pib", "crecimiento", "empleo", "desempleo", "actividad"],
}
# Señales de dirección para el PESO (bullish_cop=True => USD/COP baja)
COP_UP   = ["sube el peso", "fortalece", "petróleo sube", "brent avanza", "exportaciones crecen",
            "exportaciones suben", "superávit", "entrada de capital", "mejora la balanza"]
COP_DOWN = ["dólar fuerte", "fortalece al dólar", "déficit", "fuga de capital", "tasas altas",
            "fed mantiene", "riesgo", "incertidumbre", "preocupa", "caída del petróleo"]

def _kw_topic(text: str) -> str:
    t = text.lower()
    for topic, kws in TOPIC_RULES.items():
        if any(k in t for k in kws):
            return topic
    return "other"

def _kw_keywords(text: str, k: int = 4) -> list[str]:
    words = re.findall(r"[a-záéíóúñ]{5,}", text.lower())
    stop = {"sobre","entre","desde","entre","según","entreg","banco","república","colombia","mercados"}
    seen, out = set(), []
    for w in words:
        if w in stop or w in seen:
            continue
        seen.add(w); out.append(w)
        if len(out) >= k:
            break
    return out

def _kw_direction(text: str) -> tuple[bool | None, str]:
    t = text.lower()
    up = sum(1 for s in COP_UP if s in t)
    down = sum(1 for s in COP_DOWN if s in t)
    if up > down:  return True,  "señales de fortalecimiento del peso (USD/COP baja)"
    if down > up:  return False, "señales de debilitamiento del peso (USD/COP sube)"
    return None, "señal ambigua"

def _kw_severity(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ["banrep", "fed", "déficit", "fiscal", "tasa"]):
        return "high"
    if any(w in t for w in ["petróleo", "exportaci", "inflación"]):
        return "medium"
    return "low"

@dataclass
class Enriched:
    article: object
    topic: str
    keywords: list
    severity: str
    bullish_cop: bool | None   # True=COP se fortalece (USD/COP baja)
    reasoning: str

def enrich_heuristic(arts) -> list[Enriched]:
    out = []
    for a in arts:
        text = f"{a.title}. {getattr(a, 'summary', '')}"
        b, why = _kw_direction(text)
        out.append(Enriched(a, _kw_topic(text), _kw_keywords(text),
                            _kw_severity(text), b, why))
    return out

enriched, enrich_mode = [], "heuristica"
try:
    import os
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("sin OPENAI_API_KEY")
    from cop_fx.analysis.news_analyzer import NewsAnalyzer
    res = NewsAnalyzer().analyze_batch(articles)   # usa OpenAI vía cop_fx.llm
    enriched = [Enriched(r.article, r.topic, getattr(r, "keywords", []),
                         r.severity, r.bullish_cop, r.reasoning) for r in res]
    enrich_mode = "LLM OpenAI (NewsAnalyzer)"
except Exception as exc:
    print(f"[fallback] enriquecimiento heurístico ({exc}).")
    enriched = enrich_heuristic(articles)

df_news = pd.DataFrame([{
    "title": e.article.title[:70],
    "topic": e.topic,
    "severity": e.severity,
    "bullish_cop": {True: "COP↑ (USD↓)", False: "COP↓ (USD↑)", None: "ambiguo"}[e.bullish_cop],
    "keywords": ", ".join(e.keywords),
} for e in enriched])
print(f"Modo enriquecimiento: {enrich_mode}\n")
df_news

## Paso 3 — Grafo relacional topics ↔ noticias

Tu idea original: un grafo que conecte tópicos con noticias para *ver* de qué está hablando el día. Lo construimos con `networkx`. Nodos = tópicos + noticias; aristas = pertenencia. El tamaño del nodo-tópico crece con cuántas noticias lo tocan.

In [ ]:
# --- Paso 3: grafo topics <-> noticias -------------------------------------
try:
    import networkx as nx
    import matplotlib.pyplot as plt

    G = nx.Graph()
    topic_count = {}
    for i, e in enumerate(enriched):
        nid = f"n{i}"
        G.add_node(nid, kind="news", label=e.article.title[:40])
        G.add_node(e.topic, kind="topic", label=e.topic)
        G.add_edge(e.topic, nid)
        topic_count[e.topic] = topic_count.get(e.topic, 0) + 1

    topics = [n for n, d in G.nodes(data=True) if d["kind"] == "topic"]
    news   = [n for n, d in G.nodes(data=True) if d["kind"] == "news"]

    plt.figure(figsize=(11, 7))
    pos = nx.spring_layout(G, k=0.9, seed=42)
    nx.draw_networkx_edges(G, pos, alpha=0.3)
    nx.draw_networkx_nodes(G, pos, nodelist=news, node_color="#9ecae1",
                           node_size=320, label="noticia")
    nx.draw_networkx_nodes(G, pos, nodelist=topics, node_color="#fc9272",
                           node_size=[700 + 400 * topic_count[t] for t in topics],
                           label="tópico")
    nx.draw_networkx_labels(G, pos,
        labels={t: t for t in topics}, font_size=9, font_weight="bold")
    plt.title(f"Topics ↔ noticias — {TODAY}")
    plt.legend(scatterpoints=1); plt.axis("off"); plt.tight_layout(); plt.show()

    print("Noticias por tópico:")
    for t, c in sorted(topic_count.items(), key=lambda x: -x[1]):
        print(f"  {c:>2}  {t}")
except Exception as exc:
    print(f"[skip] grafo no disponible ({exc}). Conteo por tópico:")
    from collections import Counter
    for t, c in Counter(e.topic for e in enriched).most_common():
        print(f"  {c:>2}  {t}")

## Paso 4 — Serie de tiempo USD/COP (30 días)

Bajamos el histórico del dólar. Primario: **Stooq** (CSV, sin API key). Respaldo: el `FXFetcher` del proyecto. Último respaldo: un *random walk* sintético — así el forecast del paso 5 siempre tiene con qué correr.

Esto también es Bronze→Silver: una serie `(ds, y)`, numérica y aburrida. *Sin LLM.*

In [ ]:
# --- Paso 4: serie USD/COP 30 días -----------------------------------------
def _from_stooq() -> pd.DataFrame:
    import httpx, io
    url = "https://stooq.com/q/d/l/?s=usdcop&i=d"
    r = httpx.get(url, timeout=20, follow_redirects=True)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    if "Close" not in df.columns or df.empty:
        raise RuntimeError("Stooq sin datos")
    df = df.rename(columns={"Date": "ds", "Close": "y"})[["ds", "y"]]
    df["ds"] = pd.to_datetime(df["ds"])
    return df.dropna()

def _synthetic_fx(n=45) -> pd.DataFrame:
    rng = np.random.default_rng(7)
    days = pd.bdate_range(end=TODAY, periods=n)
    steps = rng.normal(0, 18, n)
    steps[-3:] -= 25  # leve sesgo bajista reciente para que el producto sea visible
    y = 4000 + np.cumsum(steps)
    return pd.DataFrame({"ds": days, "y": y})

fx, fx_mode = None, "stooq"
try:
    fx = _from_stooq()
except Exception as exc1:
    try:
        from cop_fx.data.fx_fetcher import FXFetcher
        fx = FXFetcher().fetch(lookback_days=45)
        fx_mode = "FXFetcher"
        if fx is None or fx.empty:
            raise RuntimeError("vacío")
    except Exception as exc2:
        fx_mode = "sintetico"
        print(f"[fallback] Stooq ({exc1}) y FXFetcher ({exc2}) fallaron. FX sintético.")
        fx = _synthetic_fx()

fx = fx.sort_values("ds").tail(30).reset_index(drop=True)
last_price = float(fx["y"].iloc[-1])
prev_price = float(fx["y"].iloc[-2])
print(f"Modo FX: {fx_mode}  |  {len(fx)} días  |  último USD/COP: {last_price:,.2f} "
      f"({'+' if last_price>=prev_price else ''}{last_price-prev_price:,.2f} vs ayer)")
fx.tail()

In [ ]:
# --- Paso 4b: ver la serie -------------------------------------------------
import matplotlib.pyplot as plt
plt.figure(figsize=(11, 4))
plt.plot(fx["ds"], fx["y"], marker="o", ms=3, color="#2c7fb8")
plt.title(f"USD/COP — últimos {len(fx)} días ({fx_mode})")
plt.ylabel("COP por USD"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Paso 5 — Forecasting

Corremos los modelos del proyecto (**Prophet** y **ARIMA**) y un **ensemble**. Para la dirección solo nos importa el **signo**: ¿el forecast apunta arriba o abajo del último valor real? Si los modelos no están instalados, caemos a una tendencia lineal simple.

In [ ]:
# --- Paso 5: forecast ------------------------------------------------------
HORIZON = 5
forecast_df, fc_mode = None, "proyecto"

def _simple_trend(df, h=HORIZON):
    x = np.arange(len(df)); y = df["y"].values
    slope, intercept = np.polyfit(x, y, 1)
    fut_x = np.arange(len(df), len(df) + h)
    fut = pd.bdate_range(start=df["ds"].iloc[-1], periods=h + 1)[1:]
    yhat = slope * fut_x + intercept
    return pd.DataFrame({"ds": fut, "yhat": yhat})

try:
    from cop_fx.timeseries.models import (
        ProphetForecaster, ARIMAForecaster, ensemble_forecast)
    results = []
    for name, mk in [("prophet", lambda: ProphetForecaster()),
                     ("arima", lambda: ARIMAForecaster(order=(2, 1, 2)))]:
        try:
            results.append(mk().fit_predict(fx, horizon_days=HORIZON))
        except Exception as e:
            print(f"  [skip {name}] {e}")
    if not results:
        raise RuntimeError("ningún modelo corrió")
    forecast_df = ensemble_forecast(results) if len(results) > 1 else results[0].forecast
    forecast_df = forecast_df.rename(columns={c: "yhat" for c in forecast_df.columns
                                              if c.lower() in ("yhat", "ensemble", "mean")})
    if "yhat" not in forecast_df.columns:
        forecast_df["yhat"] = forecast_df.select_dtypes("number").iloc[:, 0]
except Exception as exc:
    fc_mode = "tendencia lineal"
    print(f"[fallback] modelos del proyecto no disponibles ({exc}). Tendencia lineal.")
    forecast_df = _simple_trend(fx)

target = float(forecast_df["yhat"].iloc[-1])
delta = target - last_price
ts_dir = "down" if delta < -1 else "up" if delta > 1 else "neutral"
print(f"\nForecast ({fc_mode}, +{HORIZON}d): {target:,.2f}  "
      f"(Δ {delta:+,.2f} vs hoy {last_price:,.2f})")
print(f"Dirección de la SERIE: {ts_dir.upper()}")

plt.figure(figsize=(11, 4))
plt.plot(fx["ds"], fx["y"], marker="o", ms=3, label="real", color="#2c7fb8")
plt.plot(forecast_df["ds"], forecast_df["yhat"], marker="s", ms=4,
         ls="--", label="forecast", color="#de2d26")
plt.axhline(last_price, color="gray", ls=":", alpha=0.6)
plt.title(f"USD/COP + forecast {HORIZON}d ({fc_mode})")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Paso 6 — Señales y reconciliación

El corazón del producto. Dos señales independientes:

- **Señal-noticias** — suma de `bullish_cop` ponderada por `severity`. ¿Lo que se dijo hoy empuja al peso arriba o abajo?
- **Señal-serie** — el signo del forecast (paso 5).

La **reconciliación** las cruza: si concuerdan → confianza alta; si divergen → conflicto, confianza baja o abstención. Aquí es donde "se relaciona la noticia con el precio".

In [ ]:
# --- Paso 6: señales + reconciliación --------------------------------------
SEV_W = {"high": 3, "medium": 2, "low": 1}

score = 0.0
contribs = []
for e in enriched:
    if e.bullish_cop is None:
        continue
    w = SEV_W.get(e.severity, 1)
    s = (+w) if e.bullish_cop else (-w)   # + => COP se fortalece => USD/COP baja
    score += s
    contribs.append((s, e.severity, e.article.title[:55]))

# score > 0  => noticias empujan USD/COP hacia abajo (down)
news_dir = "down" if score > 0.5 else "up" if score < -0.5 else "neutral"

print(f"Señal NOTICIAS: {news_dir.upper()}  (score ponderado = {score:+.1f})")
print(f"Señal SERIE   : {ts_dir.upper()}  (Δ forecast = {delta:+,.2f})\n")

print("Aportes por noticia (+ empuja USD↓ / COP fuerte):")
for s, sev, t in sorted(contribs, key=lambda x: -abs(x[0])):
    print(f"  {s:+d} [{sev:>6}]  {t}")

reconciliation = "agree" if news_dir == ts_dir and news_dir != "neutral" else "diverge"
print(f"\nReconciliación: {reconciliation.upper()}")

## Paso 7 — El producto: `DirectionalCall`

Todo converge en una sola decisión: dirección + confianza + razonamiento + el contra-argumento. Si las señales divergen o son débiles, el sistema **se abstiene** (`neutral`) — la abstención es una capacidad, no una falla.

In [ ]:
# --- Paso 7: DirectionalCall (el producto) ---------------------------------
from dataclasses import dataclass, field as dfield

@dataclass
class DirectionalCall:
    date: str
    direction: str            # down / up / neutral
    confidence: float
    horizon_days: int
    news_signal: str
    ts_signal: str
    reconciliation: str
    rationale: str
    devils_advocate: str
    caveats: list = dfield(default_factory=list)

# Lógica de decisión + calibración de confianza
if reconciliation == "agree":
    direction = news_dir
    confidence = round(min(0.9, 0.6 + 0.05 * abs(score)), 2)
    rationale = (f"Noticias y serie coinciden en '{direction}'. "
                 f"El score de noticias ({score:+.1f}) y el forecast "
                 f"(Δ {delta:+,.2f}) apuntan en la misma dirección.")
elif news_dir == "neutral" and ts_dir != "neutral":
    direction = ts_dir
    confidence = 0.45
    rationale = ("Las noticias del día son ambiguas; la única señal direccional "
                 f"viene de la serie ({ts_dir}). Confianza moderada-baja.")
elif ts_dir == "neutral" and news_dir != "neutral":
    direction = news_dir
    confidence = 0.45
    rationale = ("La serie está plana; la señal viene de las noticias "
                 f"({news_dir}). Confianza moderada-baja.")
else:
    direction = "neutral"
    confidence = 0.3
    rationale = (f"Señales en CONFLICTO: noticias dicen '{news_dir}', serie dice "
                 f"'{ts_dir}'. El sistema se abstiene en lugar de forzar una predicción.")

devils = {
    "down": "Una noticia-shock del lado USD (Fed más dura, risk-off global) podría "
            "revertir el fortalecimiento del peso pese a la tendencia local.",
    "up":   "Un repunte fuerte del Brent o entrada de capital extranjero podría "
            "frenar la depreciación que sugieren las señales.",
    "neutral": "Abstenerse tiene costo de oportunidad: si una señal era genuina, "
               "perdemos la llamada correcta por exceso de cautela.",
}[direction]

caveats = [f"Datos de noticias en modo: {source_mode}/{enrich_mode}.",
           f"Serie FX en modo: {fx_mode}; forecast: {fc_mode}.",
           "Walking skeleton — sin LLM adjudicador ni RAG todavía."]

call = DirectionalCall(
    date=str(TODAY), direction=direction, confidence=confidence,
    horizon_days=HORIZON, news_signal=news_dir, ts_signal=ts_dir,
    reconciliation=reconciliation, rationale=rationale,
    devils_advocate=devils, caveats=caveats)

ARROW = {"down": "▼ USD/COP BAJA (peso se fortalece)",
         "up": "▲ USD/COP SUBE (peso se debilita)",
         "neutral": "■ ABSTENCIÓN (señales no concluyen)"}

print("=" * 64)
print(f"  PRODUCTO — USD/COP {call.date}")
print("=" * 64)
print(f"  Dirección  : {ARROW[call.direction]}")
print(f"  Confianza  : {call.confidence:.0%}   Horizonte: {call.horizon_days}d")
print(f"  Noticias   : {call.news_signal}   Serie: {call.ts_signal}   "
      f"→ {call.reconciliation}")
print("-" * 64)
print(f"  Razón      : {call.rationale}")
print(f"  Contra     : {call.devils_advocate}")
print("-" * 64)
for c in call.caveats:
    print(f"  ⚠ {c}")
print("=" * 64)

---

## Qué acabas de ver — y qué sigue

Este notebook ejecutó **a mano** el pipeline completo: noticias → enriquecimiento → grafo → serie → forecast → reconciliación → `DirectionalCall`. Eso es el producto.

**Lo que falta para que sea el sistema real** (en este orden):

1. **LLM real en el paso 2 y 6.** Hoy es heurística de keywords. El `NewsAnalyzer` (con API key) y un nodo adjudicador con `with_structured_output` reemplazan eso — *ahí entra LangGraph* (ver `docs/arquitectura.md`, sección 11).
2. **Persistir Bronze/Silver/Gold.** El extractor (`cop_fx.ingestion.news`) ya guarda Bronze+Silver; falta Gold y la tabla `predictions` para backtesting.
3. **El grafo agentic.** router → orchestrator/`Send` → workers → adjudicador (`evaluator.py`). Solo cuando una sola llamada ya no alcance.
4. **Infra.** Cloud Run Jobs + Scheduler, una corrida al día. Fracciones de centavo.

**Para Claude Code:** *"Lee `docs/arquitectura.md`. Implementa la Fase 1 respetando las capas Bronze→Silver de la sección 11."*